# Washington State — a century of river flow (1895–2025)

A **detailed, state-wide** companion to `wa_flow_yoy.ipynb` (which profiled a single
basin). This report reconstructs the **year-by-year monthly streamflow** of every
Washington HUC4 basin across the *entire* nClimGrid record — **1895–2025, 131 years** —
and rolls it up to a state view: how the seasonal hydrograph has shifted, whether the
spring freshet is trending, which years were the deepest droughts, and how the timing of
peak flow has drifted over the century.

It drives this repo's own offline engine (`src.historical_flow` +
`src.monthly_flow.disaggregate_monthly`) with the **license-free NOAA nClimGrid-Monthly**
provider (`tools.nclimgrid_flow.NClimGridClimateProvider`) — U.S. federal public domain,
free to sell with attribution (roadmap #60). PRISM is deliberately *not* used here.

> **What the model does — and doesn't.** NHDPlus HR gives each reach a **mean-annual**
> discharge (`QIncrAMA`, cfs); the snow-aware model redistributes that fixed annual total
> across the 12 months using each year's real precip + temperature, then accumulates
> downstream. It **conserves each reach's annual mean by construction**, so the
> year-over-year signal lives entirely in **seasonal timing and peak magnitude**, not the
> annual total. Read it as *relative* climate-driven seasonality, not a gauge record.

**Kernel:** select **Python (hydro-art)** (the project `.venv`).

## 1 · Setup & configuration

In [ ]:
import pickle
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.crs import INTERNAL_CRS
from src.monthly_flow import MONTH_ABBR, disaggregate_monthly
from src.historical_flow import normalize_years
from src import flow_metrics as fm
from tools.nclimgrid_flow import NClimGridClimateProvider, DEFAULT_ROOT
from tools.render_common import STATE_HUC4

# --- configuration -----------------------------------------------------
STATE = "Washington"
WA_HUC4 = STATE_HUC4[STATE]          # ['1701','1702','1703','1707','1708','1710','1711']
START, END = 1895, 2025              # full nClimGrid record (last complete year)
ROOT = Path(DEFAULT_ROOT)            # holds nclimgrid/nclimgrid_{prcp,tavg}.nc
CLIMATE = "nclimgrid"               # public-domain, sellable

YEARS = list(normalize_years(range(START, END + 1), latest=END))
plt.rcParams.update({"figure.facecolor": "#07080c", "axes.facecolor": "#0b0d13",
                     "savefig.facecolor": "#07080c"})

print(f"repo        : {REPO}")
print(f"state       : {STATE}  HUC4 {WA_HUC4}")
print(f"years       : {YEARS[0]}-{YEARS[-1]}  ({len(YEARS)} yrs)")
print(f"climate     : {CLIMATE}  ({ROOT}/nclimgrid)")
print(f"internal CRS: {INTERNAL_CRS}")

## 2 · Basin names (from the WBD)

Read the authoritative HUC4 names straight from the Watershed Boundary Dataset so the
panels are self-labelling (falls back to the bare code if the WBD isn't on disk).

In [ ]:
import glob
BASIN_NAMES = {}
try:
    import geopandas as gpd
    for gdb in sorted(glob.glob(str(REPO / "datasets" / "wbd" / "**" / "*.gdb"), recursive=True)):
        try:
            g = gpd.read_file(gdb, layer="WBDHU4", columns=["huc4", "name"])
        except Exception:
            continue
        for _, r in g.iterrows():
            if str(r["huc4"]) in WA_HUC4:
                BASIN_NAMES[str(r["huc4"])] = r["name"]
except Exception as exc:
    print("WBD name lookup skipped:", exc)

for c in WA_HUC4:
    BASIN_NAMES.setdefault(c, f"HUC4 {c}")
    print(f"  {c}  {BASIN_NAMES[c]}")

## 3 · Load the per-basin river networks (cached)

Reuses the topology + centroids `tools/render_state_yoy.py` staged
(`output/_yoy_net_<huc4>.pkl`): each reach's incremental annual flow (`QIncrAMA`), the
`HydroSeq`/`DnHydroSeq` connectivity, and a representative lon/lat in EPSG:4326. We
concatenate all seven basins into one coordinate array so the climate grids are read
**once per year for the whole state** (then sliced back per basin for the downstream
accumulation, which is basin-local).

In [ ]:
NET = {}
for c in WA_HUC4:
    cache = REPO / f"output/_yoy_net_{c}.pkl"
    ids, q_incr, hydroseq, dnhydroseq, lon, lat = pickle.loads(cache.read_bytes())
    NET[c] = dict(ids=ids, q_incr=q_incr.astype(np.float64), hydroseq=hydroseq,
                  dnhydroseq=dnhydroseq, lon=lon.astype(np.float64), lat=lat.astype(np.float64))

lon_all = np.concatenate([NET[c]["lon"] for c in WA_HUC4])
lat_all = np.concatenate([NET[c]["lat"] for c in WA_HUC4])
SLICES, start = {}, 0
for c in WA_HUC4:
    n = len(NET[c]["lon"]); SLICES[c] = slice(start, start + n); start += n

print(f"{len(lon_all):,} reaches across {len(WA_HUC4)} basins")
for c in WA_HUC4:
    print(f"  {c} {BASIN_NAMES[c][:34]:34s} {len(NET[c]['lon']):>8,} reaches")

## 4 · Reconstruct year-by-year monthly flow — the heavy step (cached)

For each of the 131 years we sample that year's 12 monthly precip + temperature grids at
**every** Washington reach (each nClimGrid band read once), then run the shared
`disaggregate_monthly` model per basin. We keep, per basin: the **outlet** reach's monthly
flow for every year (`[Y, 12]`), and the record-mean flow per reach (for the map).

> **Runtime:** the first run reads ~3,100 NetCDF bands off the NAS — expect **~15–30 min**.
> The result is cached to `output/_wa_state_report_{START}_{END}.pkl`, so every rerun is
> instant. Delete that file to recompute.

In [ ]:
CACHE = REPO / f"output/_wa_state_report_{START}_{END}.pkl"

if CACHE.exists():
    STATE_SERIES = pickle.loads(CACHE.read_bytes())
    print(f"loaded cached series: {CACHE.name}")
else:
    provider = NClimGridClimateProvider(ROOT, lon_all, lat_all)
    acc = {c: dict(sum_mean=np.zeros(len(NET[c]["lon"])), outlet=None, monthly=[])
           for c in WA_HUC4}
    for yi, year in enumerate(YEARS):
        clim = provider.climate_for_year(year)          # [N,12] precip + temp, one read/band
        for c in WA_HUC4:
            sl, net = SLICES[c], NET[c]
            flow = disaggregate_monthly(
                clim.precip_mm[sl], clim.temp_c[sl],
                net["q_incr"], net["hydroseq"], net["dnhydroseq"],
            )                                            # [n,12] accumulated cfs
            m = flow.mean(axis=1)
            a = acc[c]
            a["sum_mean"] += m
            if a["outlet"] is None:                      # outlet = max-accumulation reach
                a["outlet"] = int(m.argmax())            # (drainage-dominated, year-stable)
            a["monthly"].append(flow[a["outlet"]])
        print(f"  {year}  ({yi + 1}/{len(YEARS)})", flush=True)

    STATE_SERIES = {"years": YEARS, "basins": {}}
    for c in WA_HUC4:
        a = acc[c]
        STATE_SERIES["basins"][c] = dict(
            outlet=a["outlet"],
            outlet_monthly=np.asarray(a["monthly"], dtype=np.float64),   # [Y,12]
            mean_flow_per_reach=(a["sum_mean"] / len(YEARS)).astype(np.float32),
        )
    CACHE.write_bytes(pickle.dumps(STATE_SERIES))
    print(f"wrote {CACHE.name}")

YEARS = STATE_SERIES["years"]
BAS = STATE_SERIES["basins"]
print("basins:", list(BAS))

## 5 · A state-wide flow index

We form two views:

- **Combined outlet flow** — the sum of the seven basin-outlet hydrographs, a single
  state-scale monthly series `state[year] -> [12]`. (Basins nest partly via the Columbia,
  so treat this as a consistent *index*, not a mass balance.)
- **Per-basin outlets** — kept separately for the small-multiples panel.

In [ ]:
YEAR_IDX = {y: i for i, y in enumerate(YEARS)}
# state combined-outlet monthly series {year: [12]}
state = {y: sum(BAS[c]["outlet_monthly"][YEAR_IDX[y]] for c in WA_HUC4) for y in YEARS}
state_mat = np.asarray([state[y] for y in YEARS])            # [Y,12]

peak_flow = state_mat.max(axis=1)                            # [Y] seasonal peak
peak_month = state_mat.argmax(axis=1)                        # [Y] 0-based
annual_total = state_mat.sum(axis=1)                         # [Y] (index of wetness)
print(f"combined-outlet peak flow spread: {peak_flow.min():,.0f}..{peak_flow.max():,.0f} cfs")
print(f"driest year (by total): {YEARS[int(annual_total.argmin())]}, "
      f"wettest: {YEARS[int(annual_total.argmax())]}")

## 6 · Where the water is — statewide reach map

Every Washington reach centroid, colored by log record-mean flow. The bright spines are
the mainstems (Columbia, Snake, Yakima, Skagit, …); red stars mark each basin's outlet.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
palette = cm.viridis
for c in WA_HUC4:
    mfr = BAS[c]["mean_flow_per_reach"]
    ax.scatter(NET[c]["lon"], NET[c]["lat"], c=np.log10(mfr + 1.0),
               s=1.5, cmap=palette, linewidths=0, vmin=0, vmax=5)
    o = BAS[c]["outlet"]
    ax.scatter(NET[c]["lon"][o], NET[c]["lat"][o], marker="*", s=220,
               edgecolor="white", facecolor="red", zorder=6)
ax.set_aspect(1 / np.cos(np.deg2rad(lat_all.mean())))
ax.set_title(f"{STATE} — {len(lon_all):,} reaches by log record-mean flow (1895-2025)",
             color="#e8ecf7")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
sm = cm.ScalarMappable(cmap=palette); sm.set_clim(0, 5)
cb = fig.colorbar(sm, ax=ax, shrink=0.6, label="log10 mean flow (cfs)")
for a in (ax.xaxis.label, ax.yaxis.label, cb.ax.yaxis.label):
    a.set_color("#9fb3d0")
ax.tick_params(colors="#7f93b0"); cb.ax.tick_params(colors="#7f93b0")
plt.tight_layout(); plt.show()

## 7 · How the seasonal hydrograph shifted, decade by decade

Each line is a **decadal mean** monthly hydrograph of the combined outlet, cool→warm from
the 1890s to the 2020s. Watch the spring-freshet peak change height and drift in time.

In [ ]:
decades = {}
for y in YEARS:
    decades.setdefault((y // 10) * 10, []).append(state[y])
dkeys = sorted(decades)

fig, ax = plt.subplots(figsize=(11, 5.5))
cmap = cm.plasma
dd = max(len(dkeys) - 1, 1)
for i, d in enumerate(dkeys):
    mean_hg = np.mean(decades[d], axis=0)
    ax.plot(MONTH_ABBR, mean_hg, marker="o", ms=3,
            color=cmap(i / dd), label=f"{d}s")
ax.set_ylabel("combined-outlet flow (cfs)", color="#9fb3d0")
ax.set_title(f"{STATE} — decadal mean monthly hydrograph", color="#e8ecf7")
ax.legend(ncol=4, fontsize=8, facecolor="#0b0d13", labelcolor="#cdd7ea")
ax.grid(alpha=0.2); ax.tick_params(colors="#7f93b0")
plt.tight_layout(); plt.show()

## 8 · The long record — is the spring peak trending?

Combined-outlet **peak-month flow** for every year 1895–2025, with a **30-year rolling
normal** and a **Mann-Kendall** trend test + **Sen's slope** (both from
`src.flow_metrics`). Mann-Kendall is non-parametric (robust to the heavy interannual
variance); Sen's slope is the median pairwise slope.

In [ ]:
mk = fm.mann_kendall(peak_flow)
slope = fm.sens_slope(peak_flow)
normals = fm.rolling_normals(peak_flow, window=min(30, len(peak_flow)))

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.plot(YEARS, peak_flow, color="#38bdf8", lw=1.0, alpha=0.8, label="annual peak-month flow")
for nrm in normals:
    ax.hlines(nrm.mean, YEARS[nrm.start], YEARS[nrm.end], color="#f59e0b", lw=2.5, alpha=0.9)
ax.plot([], [], color="#f59e0b", lw=2.5, label="30-yr rolling normal")
x = np.asarray(YEARS, float)
ax.plot(x, slope * (x - x[0]) + peak_flow[0], "--", color="#e5e7eb", lw=1.2,
        label=f"Sen's slope {slope:+,.0f} cfs/yr")
ax.set_ylabel("peak-month flow (cfs)", color="#9fb3d0")
ax.set_title(f"{STATE} combined outlet — peak flow, 1895-2025  "
             f"[Mann-Kendall: {mk.trend}, tau={mk.tau:+.2f}, p={mk.p:.3f}]",
             color="#e8ecf7")
ax.legend(fontsize=9, facecolor="#0b0d13", labelcolor="#cdd7ea")
ax.grid(alpha=0.2); ax.tick_params(colors="#7f93b0")
plt.tight_layout(); plt.show()
print(f"Mann-Kendall: S={mk.S}, tau={mk.tau:+.3f}, p={mk.p:.4f} -> {mk.trend}")
print(f"Sen's slope : {slope:+,.1f} cfs/yr  ({slope * (len(YEARS) - 1):+,.0f} cfs over record)")

## 9 · The "typical year" and its envelope

Climatological hydrograph: the mean monthly combined-outlet flow (the expected shape) with
a **P10–P90 band** across all 131 years — what a normal year looks like, and how wide any
given month can swing.

In [ ]:
mean_month = state_mat.mean(axis=0)
p10 = np.percentile(state_mat, 10, axis=0)
p90 = np.percentile(state_mat, 90, axis=0)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(MONTH_ABBR, mean_month, marker="o", color="#38bdf8", label="mean (typical year)")
ax.fill_between(MONTH_ABBR, p10, p90, alpha=0.25, color="#38bdf8", label="P10-P90 across years")
ax.set_ylabel("combined-outlet flow (cfs)", color="#9fb3d0")
ax.set_title(f"{STATE} — typical-year hydrograph with P10-P90 envelope", color="#e8ecf7")
ax.legend(fontsize=9, facecolor="#0b0d13", labelcolor="#cdd7ea")
ax.grid(alpha=0.2); ax.tick_params(colors="#7f93b0")
plt.tight_layout(); plt.show()

## 10 · Deepest droughts vs. biggest water years

Years ranked by total combined-outlet flow. The three driest and three wettest years'
hydrographs, against the typical year (dashed) — the shape of a drought vs. a deluge.

In [ ]:
order = np.argsort(annual_total)
driest = [YEARS[i] for i in order[:3]]
wettest = [YEARS[i] for i in order[-3:][::-1]]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, group, title, col in ((a1, driest, "3 driest years", cm.autumn),
                              (a2, wettest, "3 wettest years", cm.winter)):
    ax.plot(MONTH_ABBR, mean_month, "--", color="#8899aa", lw=1.2, label="typical year")
    for i, y in enumerate(group):
        ax.plot(MONTH_ABBR, state[y], marker="o", ms=3, color=col(0.2 + 0.3 * i), label=str(y))
    ax.set_title(f"{STATE} — {title}", color="#e8ecf7")
    ax.legend(fontsize=9, facecolor="#0b0d13", labelcolor="#cdd7ea")
    ax.grid(alpha=0.2); ax.tick_params(colors="#7f93b0")
a1.set_ylabel("combined-outlet flow (cfs)", color="#9fb3d0")
plt.tight_layout(); plt.show()
print("driest :", driest)
print("wettest:", wettest)

## 11 · Has peak flow drifted earlier? (center of timing)

The **center of timing** — the flow-weighted mean month — for each year, with a
Mann-Kendall trend. A downward trend means the freshet arrives earlier (a classic
warming-driven snowmelt signal).

In [ ]:
cot = np.array([fm.center_of_timing(state[y]) for y in YEARS])   # 1-based month centroid
mk_t = fm.mann_kendall(cot)
slope_t = fm.sens_slope(cot)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(YEARS, cot, color="#a78bfa", lw=1.0, alpha=0.85, label="center of timing")
x = np.asarray(YEARS, float)
ax.plot(x, slope_t * (x - x[0]) + cot[0], "--", color="#e5e7eb", lw=1.2,
        label=f"Sen's slope {slope_t:+.3f} month/yr")
ax.set_ylabel("center of timing (month)", color="#9fb3d0")
ax.set_yticks(range(1, 13)); ax.set_yticklabels(MONTH_ABBR)
ax.set_title(f"{STATE} — flow center of timing, 1895-2025  "
             f"[Mann-Kendall: {mk_t.trend}, p={mk_t.p:.3f}]", color="#e8ecf7")
ax.legend(fontsize=9, facecolor="#0b0d13", labelcolor="#cdd7ea")
ax.grid(alpha=0.2); ax.tick_params(colors="#7f93b0")
plt.tight_layout(); plt.show()
print(f"center-of-timing trend: {slope_t:+.4f} month/yr over {len(YEARS)} yrs "
      f"({slope_t * (len(YEARS) - 1):+.2f} months) -> {mk_t.trend}")

## 12 · Per-basin small multiples

Each Washington HUC4 basin's own **typical-year hydrograph** (mean ± P10–P90) plus its
peak-flow Mann-Kendall verdict. Compare the snowmelt-dominated interior basins (single
late-spring peak) against the rain-driven coastal ones (winter peak).

In [ ]:
ncol = 3
nrow = int(np.ceil(len(WA_HUC4) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(14, 3.4 * nrow), sharex=True)
axes = np.asarray(axes).ravel()
for ax, c in zip(axes, WA_HUC4):
    mo = BAS[c]["outlet_monthly"]                       # [Y,12]
    mean_c = mo.mean(axis=0)
    p10c, p90c = np.percentile(mo, 10, axis=0), np.percentile(mo, 90, axis=0)
    pk = mo.max(axis=1)
    mk_c = fm.mann_kendall(pk)
    ax.plot(MONTH_ABBR, mean_c, color="#38bdf8", marker="o", ms=2)
    ax.fill_between(MONTH_ABBR, p10c, p90c, alpha=0.22, color="#38bdf8")
    ax.set_title(f"{c} {BASIN_NAMES[c][:24]}\npeak trend: {mk_c.trend} (p={mk_c.p:.2f})",
                 color="#e8ecf7", fontsize=9)
    ax.grid(alpha=0.2); ax.tick_params(colors="#7f93b0", labelsize=7)
for ax in axes[len(WA_HUC4):]:
    ax.set_visible(False)
fig.suptitle(f"{STATE} — per-basin typical-year hydrographs (outlet, 1895-2025)",
             color="#e8ecf7", y=1.005)
plt.tight_layout(); plt.show()

## 13 · Summary

- **State index built** from all seven WA HUC4 basins × 131 years of real nClimGrid
  climate, driving the repo's own offline disaggregation engine.
- The **long-record trend** (§8) and **center-of-timing** (§11) panels are the headline
  climate-signal read; the **per-basin multiples** (§12) show how the snowmelt vs.
  rain regimes differ across the state.

**Caveats (keep honest).** Flow is *modeled, not gauged* — annual means are conserved by
construction, so only **seasonality and peaks** carry the year-over-year signal. The
combined-outlet series is an **index** (basins nest via the Columbia), not a mass balance.
Climate is public-domain nClimGrid, so any figure here is **sellable with attribution**.

**Extend it.** Swap `STATE` to another supported region (Oregon / California / Idaho) —
the same cells work off that state's `output/_yoy_net_*.pkl`. For gauge validation, join
observed USGS discharge via `tools/nwis_gauge.py` and `src.flow_metrics.validate`.